In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException
#from webdriver_manager.chrome import ChromeDriverManager

import sqlite3
import time
import pandas as pd

def extract_table_data(table):
    table_data = []
    rows = table.find_elements(By.TAG_NAME, "tr")

    for row in rows:
        columns = row.find_elements(By.TAG_NAME, "td")
        row_data = [column.text for column in columns]  
        table_data.append(row_data)

    return pd.DataFrame(table_data)

def get_player_ranking(player_data):
    
    player_name = player_data['PLAYER']
    
    if player_data['ATP RANKING']:
        return player_name, f"ATP ranking: {player_data['ATP RANKING']}"
    elif player_data['ITF RANKING']:
        return player_name, f"ITF ranking: {player_data['ITF RANKING']}"
    elif player_data['WTN'] != '-':
        return player_name, f"WTN: {player_data['WTN']}"
    elif player_data['NATIONAL RANKING']:
        return player_name, f"National ranking: {player_data['NATIONAL RANKING']}"
    else:
        return player_name, 'No ranking'
    
def insert_tournament(conn, tournament_data):
    curs = conn.cursor()

    query = """
    INSERT OR IGNORE INTO tTournaments (
        tournament_key, city, country, points, prize_money,
        date_started, date_ended, qualysize, qualybyes, surface, in_out
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """
    
    curs.execute(query, tournament_data)
    conn.commit()
    curs.close()



def insert_players(conn, df):
    curs = conn.cursor()

    query = """
        INSERT OR IGNORE INTO tPlayerInfo (
            player_name, country, designation, rank_type, rank_value, 
            tournament_key, acceptancelist_number, acceptancelist_type
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?);
        """
    
    for row in df.itertuples(index=False, name=None):
        curs.execute(query, row)
    conn.commit()
    curs.close()



In [4]:
conn = sqlite3.connect('itf_tournaments.db')
curs = conn.cursor()
curs.execute("PRAGMA foreign_keys=ON;")


# Create tTournaments table
curs.execute("""
CREATE TABLE IF NOT EXISTS tTournaments (
    tournament_key TEXT PRIMARY KEY,
    city TEXT,
    country TEXT,
    points INTEGER,
    prize_money INTEGER,
    date_started TEXT,
    date_ended TEXT,
    qualysize INTEGER,
    qualybyes INTEGER,
    surface TEXT,
    in_out TEXT
);
""")

# Create tPlayers table
curs.execute("""
CREATE TABLE IF NOT EXISTS tPlayerInfo (
    player_name TEXT NOT NULL,
    country TEXT,
    designation TEXT,
    rank_type TEXT,
    rank_value INTEGER,
    tournament_key TEXT,
    acceptancelist_number INTEGER,
    acceptancelist_type TEXT,
    PRIMARY KEY (player_name, tournament_key),
    FOREIGN KEY (tournament_key) REFERENCES tTournaments(tournament_key)
);
""")


In [5]:
def itf_scraper(websites):
    path = 'chromedriver.exe'
    service = Service(executable_path=path)
    driver = webdriver.Chrome(service=service)
    wait = WebDriverWait(driver, 30)
    counter = 0
    conn = sqlite3.connect('itf_tournaments.db')
    curs = conn.cursor()
    curs.execute("PRAGMA foreign_keys=ON;")
    
    for website in websites:
        tourney_key = website.split('/')[-2]

        website_draw = website + 'draws-and-results/'
        website_al = website + 'acceptance-list/'

        tournament_part = website.split('/')[-5]
        formatted_name = tournament_part.replace('-', ' ').title()

        driver.get(website)
        #if counter == 0:
        #    cookies = wait.until(EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler")))
        #    time.sleep(0.5)
        #    cookies.click()
        #    counter+=1


        qualy_size = int(driver.find_element(By.XPATH, "//*[contains(text(), 'Singles qualifying')]").text[-2:])
        driver.get(website_draw)


        driver.execute_script("window.scrollBy(0, 600);")
        time.sleep(0.8)
        dropdown = wait.until(EC.element_to_be_clickable((By.XPATH, "(//div[contains(@class, 'css-j1esxd-singleValue')])[2]")))
        dropdown.click()
        qualifying_option = wait.until(EC.element_to_be_clickable((By.XPATH, "//div[text()='Qualifying Draw']")))
        qualifying_option.click()

        time.sleep(0.5)
                
        player_l = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "drawsheet-widget__last-name")))
        player_last = []
        for lplayer in player_l:
            try:
                player_last.append(lplayer.text)
            except StaleElementReferenceException:
                break

        player_f = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "drawsheet-widget__first-name")))
        player_first = []
        for fplayer in player_f:
            try:
                player_first.append(fplayer.text)
            except StaleElementReferenceException:
                break
        
        
        countries = driver.find_elements(By.XPATH, "//span[contains(@class, 'drawsheet-widget__nationality')]")
        country_names = [country.text for country in countries]

        seen = set()
        full_names = [f"{first} {last}" for first, last in zip(player_first, player_last)
                    if f"{first} {last}" not in seen and not seen.add(f"{first} {last}")]
        
        parent_containers = driver.find_elements(By.XPATH, "//div[contains(@class, 'drawsheet-round-container is-first-round carousel__animation--drawsheet-enter-done')]")
        walkovers = 0
        for parent in parent_containers:
            walkovers += len(parent.find_elements(By.CLASS_NAME, "drawsheet-widget__alert-status-text"))
        
        byes = qualy_size - len(full_names) + walkovers

        surface_info = driver.find_element(By.ID, "ga__tournament-surface").text
        prize_money = int(driver.find_element(By.XPATH, "//span[contains(@class, 'tournament-hero__value') and contains(text(), '$')]").text[1:])
        date = driver.find_element(By.ID, "ga__tournament-dates").text
        host_country = driver.find_element(By.ID, "ga__tournament-host-nation").text.upper()

        city = " ".join(website.split('/')[-5].split('-')[1:]).upper()
        points = int(website.split('/')[-5].split('-')[0][1:3])


        start_year = website.split('/')[-2].split('-')[-2]
        date_started = date.split(' - ')[0]
        date_started = date_started + ' ' + start_year
        date_ended = date.split(' - ')[1]
        surface = surface_info.split(' - ')[0]

        if surface_info.split(' - ')[1] == 'O':
            in_out = 'Outdoor'
        else:
            in_out = 'Indoor'
        
        tournament_data = (tourney_key, city, host_country, points, prize_money, 
                           date_started, date_ended, qualy_size, byes, surface, in_out)
        

        designation_list = []
        counter = 0
        for player in player_l:
            try:
                if player.find_element(By.XPATH, "following-sibling::span"):
                    designation_span = player.find_element(By.XPATH, "following-sibling::span")
                    designation = designation_span.text.strip()
                    designation_list.append(designation)
            except:
                designation_list.append('DA')
            counter += 1
            if counter >= len(full_names):
                break



        driver.get(website_al)


        tables = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "acceptance-list")))
        columns = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "acceptance-list__title-default")))
        column_names = [column_name.text for column_name in columns]
        final_columns = []
        [final_columns.append(column) for column in column_names if column not in final_columns]
        final_columns.pop(0)
        final_columns.pop()
        final_columns.insert(0, 'PLAYER')
        final_columns.append('PRIORITY')



        dataframes = []

        if len(tables)==4:
            # There's no Junior reserved
            for i in range(3): 
                table = tables[i]
                df = extract_table_data(table)
                dataframes.append(df) 
        if len(tables)==5:
            # There IS Junior reserved
            for i in range(1, 4): 
                table = tables[i]
                df = extract_table_data(table)
                dataframes.append(df)

        if dataframes:
            cleaned_dfs = []
            for d_f in dataframes:
                d_f = d_f.drop(d_f.columns[[0, -1]], axis=1)
                d_f = d_f.drop(d_f.index[0]).reset_index(drop=True)
                d_f.columns = final_columns
                d_f['COUNTRY'] = d_f['PLAYER'].apply(lambda x: x[:3] if '\n' in x else 'N/A')
                d_f['PLAYER'] = d_f['PLAYER'].apply(lambda x: x[3:] if '\n' in x else x).str.replace('\n', '')
                cleaned_dfs.append(d_f)
                
            if cleaned_dfs:
                main_draw_al = cleaned_dfs[0]
                qualy_al = cleaned_dfs[1]
                alternate_al = cleaned_dfs[2]
            else:
                print("Error: No cleaned dataframes were found.")
        else:
            print("Error: No tables were found or extracted.")



        #combined_al = pd.concat([main_draw_al, qualy_al, alternate_al], ignore_index=True)
        combined_al_no_m = pd.concat([qualy_al, alternate_al])

        players_in_qdraw = pd.DataFrame(list(zip(full_names, designation_list, country_names)))
        players_in_qdraw.columns = ['PLAYER', 'DESIGNATION', 'COUNTRY']

        acceptance_summary = combined_al_no_m.merge(players_in_qdraw, on='PLAYER', how='inner').drop(columns=['COUNTRY_y']).rename(columns={'COUNTRY_x':'COUNTRY'})
        acceptance_summary['WTN'] = acceptance_summary['WTN'].astype(str)
        acceptance_summary['ATP RANKING'] = acceptance_summary['ATP RANKING'].astype(str)
        acceptance_summary['ITF RANKING'] = acceptance_summary['ITF RANKING'].astype(str)
        acceptance_summary['NATIONAL RANKING'] = acceptance_summary['NATIONAL RANKING'].astype(str)




        if len(acceptance_summary) != len(players_in_qdraw):
            for name in players_in_qdraw['PLAYER']:
                if name not in list(acceptance_summary['PLAYER']):
                    temp_coun = players_in_qdraw.loc[players_in_qdraw['PLAYER'] == name, 'COUNTRY'].iloc[0]
                    temp_desg = players_in_qdraw.loc[players_in_qdraw['PLAYER'] == name, 'DESIGNATION'].iloc[0]
                    new_row = {'PLAYER': name, 'COUNTRY': temp_coun, 'PRIORITY': '1', 'DESIGNATION': temp_desg}
                    acceptance_summary.loc[len(acceptance_summary)] = new_row
        #print(acceptance_summary)

        da_df = acceptance_summary[acceptance_summary['DESIGNATION'].str.contains('DA', na=False)].copy()
        last_direct_acc = da_df[da_df['ATP RANKING'].notna()].iloc[-1] if not da_df[da_df['ATP RANKING'].notna()].empty else None

        al_df = acceptance_summary[acceptance_summary['DESIGNATION'].str.contains(r'\(A\)', na=False)].copy()
        last_alt_acc = al_df.iloc[-1] if not al_df.empty else [0]

        
        df_to_players = acceptance_summary[acceptance_summary['PLAYER'] != '(Available Slot)'].reset_index(drop=True)

        df_to_players['rank_type'] = df_to_players.apply(
            lambda row: 'ATP' if row['ATP RANKING'] not in ['', None] and not isinstance(row['ATP RANKING'], float) else
                'ITF' if row['ITF RANKING'] not in ['', None] and not isinstance(row['ITF RANKING'], float) else
                'WTN' if row['WTN'] not in ['-', '', None] and not isinstance(row['WTN'], float) else
                'NATIONAL' if row['NATIONAL RANKING'] not in ['', None] and not isinstance(row['NATIONAL RANKING'], float) else
                'NONE',
            axis=1
        )

        df_to_players['rank_value'] = df_to_players.apply(
            lambda row: row['ATP RANKING'] if row['rank_type'] == 'ATP' and pd.notna(row['ATP RANKING']) else
                row['ITF RANKING'] if row['rank_type'] == 'ITF' and pd.notna(row['ITF RANKING']) else
                row['WTN'] if row['rank_type'] == 'WTN' and pd.notna(row['WTN']) else
                row['NATIONAL RANKING'] if row['rank_type'] == 'NATIONAL' and pd.notna(row['NATIONAL RANKING']) else 0,
            axis=1
        )
        

        df_to_players = df_to_players.drop(columns=['ATP RANKING', 'ITF RANKING', 'WTN', 'NATIONAL RANKING', 'PRIORITY'])
        df_to_players['tournament_key'] = tourney_key
        
        matching_indices = combined_al_no_m[combined_al_no_m['PLAYER'].isin(acceptance_summary['PLAYER'])].index
        adjusted = matching_indices+1

        if len(adjusted) < len(df_to_players):
            adjusted = list(adjusted) + [0] * (len(df_to_players) - len(adjusted))

        df_to_players['acceptancelist_number'] = adjusted
        
        status = []
        previous_value = -1  
        current_status = 'Qualifying'

        for val in df_to_players['acceptancelist_number']:
            if pd.isna(val):
                current_status = 'None'
            elif val < previous_value and current_status != 'None':
                current_status = 'Alternate'
            # Once set to Alternate, it won't go back to Qualifying
            elif current_status != 'Alternate':
                current_status = 'Qualifying'
            
            status.append(current_status)
            previous_value = val if not pd.isna(val) else previous_value

        df_to_players['acceptancelist_type'] = status

        if 'JUNIOR RANKING' in df_to_players.columns:
            df_to_players = df_to_players.drop(columns=['JUNIOR RANKING'])
        
        insert_tournament(conn, tournament_data)
        print(f'Succesfully added {tourney_key} to the database')

        insert_players(conn, df_to_players)
        print(f'Succesfully added players to database from {tourney_key}\n')

        

        print(f'Tournament: {formatted_name} - {date}')
        print(f'Qualifying draw size: {qualy_size}')
        print(f'Number of byes: {byes} \n')

        print('Last direct acceptance:')
        dir = get_player_ranking(last_direct_acc)
        print(f"{dir[0]} ({last_direct_acc['COUNTRY']}) - {dir[1]}")

        player_name = last_direct_acc['PLAYER']
        alternate_num = alternate_al.loc[alternate_al['PLAYER'] == player_name].index
        print(f'He was alternate number {alternate_num[0]+1}/{alternate_al.shape[0]}\n')

       # last_alts =  acceptance_summary[(acceptance_summary['DESIGNATION'] != '(WC)') & (acceptance_summary.isna().any(axis=1))]
       # num_last_alts = last_alts.shape[0]

        if last_alt_acc[0]!=0:
            print('Last on-site alternate (A) in:')
            alt = get_player_ranking(last_alt_acc)
            if alt[1] == 'ATP ranking: nan':
                print(f"{alt[0]} ({last_alt_acc['COUNTRY']}) - {alt[1]}")
                print(f'He was an unregistered on-site alternate\n')
            else:
                print(f"{alt[0]} ({last_alt_acc['COUNTRY']}) - {alt[1]}")

                player_name = last_alt_acc['PLAYER']
                alternate_num = alternate_al.loc[alternate_al['PLAYER'] == player_name].index
                print(f'He was alternate number {alternate_num[0]+1}/{alternate_al.shape[0]}')
                
        else:
            print('No on-site alternates (A) got in')


        #if byes != 0:
        #    print('You would\'ve gotten in!')
        #else:
        #    print(f'\n{num_last_alts} unregistered got in\n')

        
        print('---------------------------------------')
        
    conn.close()
    driver.quit()

    return


In [ ]:
#
tourneys = [
 
            'https://www.itftennis.com/en/tournament/m15-madrid/esp/2025/m-itf-esp-2025-039/',
            'https://www.itftennis.com/en/tournament/m15-bologna/ita/2025/m-itf-ita-2025-021/',
            'https://www.itftennis.com/en/tournament/m15-haren/ned/2025/m-itf-ned-2025-006/'
            

]

itf_scraper(tourneys)

In [6]:
import pandas as pd
pd.set_option('display.max_rows', None)

conn = sqlite3.connect('itf_tournaments.db')
curs = conn.cursor()
curs.execute("PRAGMA foreign_keys=ON;")
#conn.close()

In [7]:

query1 = pd.read_sql("""
                    
                WITH tWTN AS (
                SELECT
                        *
                FROM tTournaments
                WHERE date_started LIKE '%21 Apr%' 
                     OR date_started LIKE '%28 Apr%' 
                     OR date_started LIKE '%May%'
                )
                
                SELECT qualysize, count(qualysize)
                FROM tWTN
                WHERE qualybyes != 0
                GROUP BY qualysize
                ORDER BY qualysize desc
            
            ;""", conn)

query2 = pd.read_sql("""
        SELECT *
        FROM tPlayerInfo
        WHERE tournament_key = 'm-itf-esp-2024-040'
            
;""", conn)

query3 = pd.read_sql("""
        SELECT *
        FROM tTournaments
        WHERE city = 'MADRID'
;""", conn)


In [9]:
# Number of players from alternate list who got in

query2 = pd.read_sql("""
        SELECT 
                p.tournament_key, 
                t.city, 
                t.qualysize,     
                p.acceptancelist_type, 
                count(acceptancelist_type) AS count
        FROM 
                tPlayerInfo p     
        JOIN 
                tTournaments t ON p.tournament_key = t.tournament_key
        WHERE 
                --p.tournament_key LIKE '%mex%'
                designation = 'DA'
                AND acceptancelist_type = 'Alternate'
        GROUP BY 
                p.tournament_key,
                city,
                acceptancelist_type
        ORDER BY 
                count(acceptancelist_type) DESC
        LIMIT 10
            
;""", conn)
query2

,tournament_key,city,qualysize,acceptancelist_type,count
0,m-itf-esp-2025-028,SABADELL,64,Alternate,32
1,m-itf-bel-2025-004,EUPEN,64,Alternate,27
2,m-itf-tur-2025-014,ANTALYA,64,Alternate,26
3,m-itf-aut-2025-002,KRAMSACH,64,Alternate,25
4,m-itf-gre-2025-020,HERAKLION,48,Alternate,24
5,m-itf-chi-2024-004,SANTIAGO,48,Alternate,23
6,m-itf-rou-2025-004,BISTRITA,48,Alternate,23
7,m-itf-usa-2024-035,COLUMBUS,32,Alternate,23
8,m-itf-usa-2025-026,EDWARDSVILLE IL,64,Alternate,23
9,m-itf-aut-2025-003,TELFS,64,Alternate,22


In [ ]:
# FOR DELETION

conn = sqlite3.connect('itf_tournaments.db')
curs = conn.cursor()
curs.execute("PRAGMA foreign_keys=ON;")

curs.execute("""
            DELETE FROM tPlayerInfo
            WHERE tournament_key = 'm-itf-chn-2025-007'
            ;
             """)

curs.execute("""
                DELETE FROM tTournaments
                WHERE tournament_key = 'm-itf-chn-2025-007';
                """)
conn.commit()

curs.close()
conn.commit()
conn.close()

In [10]:
# Return the last player in for each tournament
query3 = pd.read_sql("""
                     WITH NonByes AS (
                     SELECT tournament_key
                     FROM tTournaments
                     WHERE date_started = '08 Sep 2025'
                     AND qualybyes == '0'
                     ),

                    RankedPlayers AS (
                        SELECT *,
                            ROW_NUMBER() OVER (
                                PARTITION BY tournament_key
                                ORDER BY acceptancelist_type ASC, acceptancelist_number DESC
                            ) AS row_num
                        FROM tPlayerInfo
                        WHERE tournament_key IN (SELECT tournament_key FROM NonByes)
                        AND (designation = 'DA' OR designation = '(A)')
                    )
                    SELECT *
                    FROM RankedPlayers
                    WHERE row_num = 1
                    ORDER BY rank_type DESC, rank_value DESC, acceptancelist_number DESC


                    
            ;""", conn)
query3

,player_name,country,designation,rank_type,rank_value,tournament_key,acceptancelist_number,acceptancelist_type,row_num
0,Davide Pierluigi Osti,ITA,DA,NONE,0,m-itf-ita-2025-022,64,Alternate,1
1,Alex Blanchar,ESP,(A),NONE,0,m-itf-esp-2025-047,33,Alternate,1
2,James Furlong,AUS,DA,NONE,0,m-itf-aus-2025-007,28,Alternate,1
3,Michael Nicholls,USA,DA,NONE,0,m-itf-ina-2025-007,24,Alternate,1
4,Oscar Houille,FRA,DA,NATIONAL,449,m-itf-fra-2025-025,22,Alternate,1
5,Sho Katayama,JPN,DA,ITF,1749,m-itf-jpn-2025-019,9,Alternate,1


In [15]:
# ALl lines below are for resetting to str for rank value

curs.execute("CREATE TABLE tPlayerInfo_backup AS SELECT * FROM tPlayerInfo;")

In [17]:
curs.execute("ALTER TABLE tPlayerInfo RENAME TO tPlayerInfo_old;")

In [20]:
curs.execute("""
             CREATE TABLE tPlayerInfo (
    player_name TEXT,
    country TEXT,
    designation TEXT,
    rank_type TEXT,
    rank_value TEXT,
    tournament_key TEXT,
    acceptancelist_number INTEGER,
    acceptancelist_type TEXT,
    PRIMARY KEY(player_name, tournament_key)
);
             """)

In [33]:
curs.execute("""
             INSERT INTO tPlayerInfo
SELECT 
    player_name,
    country,
    designation,
    rank_type,
    CAST(rank_value AS TEXT),
    tournament_key,
    acceptancelist_number,
    acceptancelist_type
FROM tPlayerInfo_backup;
             """)

In [38]:
curs.execute("DROP TABLE tPlayerInfo_old;")


In [12]:
query = pd.read_sql("SELECT * FROM sqlite_master", conn)
query

,type,name,tbl_name,rootpage,sql
0,table,tTournaments,tTournaments,2,CREATE TABLE tTournaments (\n tournament_ke...
1,index,sqlite_autoindex_tTournaments_1,tTournaments,3,None
2,table,tPlayerInfo,tPlayerInfo,4,CREATE TABLE tPlayerInfo (\n player_name TE...
3,index,sqlite_autoindex_tPlayerInfo_1,tPlayerInfo,5,None
4,table,players_temp,players_temp,315,CREATE TABLE players_temp (\n player_name T...
5,index,sqlite_autoindex_players_temp_1,players_temp,316,None
